# Inspecting a MyPTV calibration

This notebook looks at a finished calibration: how large the reprojection
error is, where it lives, which views are bad, and where the cameras and the
target actually are in space.

It works on **any** MyPTV calibration — points picked by hand, a target on a
translation stage, or a board moved freely — because everything is computed
from the camera files and the calibration points files, which every route
produces.

**Before running**, set `FOLDER` and `CAMS` in the next cell.

In [ ]:
# ---------------------------------------------------------------- settings
FOLDER = r"D:\myPTVtests\calibration_run"   # holds the camera files
POINTS = FOLDER + r"\Calibration"           # holds the <cam>_cal_points files
CAMS   = ['cam1', 'cam2', 'cam3', 'cam4']

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['figure.figsize'] = (9, 5)

from myptv.makePlots.plot_calibration import calibration_report

rep = calibration_report.from_folder(POINTS, FOLDER, CAMS)
rep

## 1. How big is the error?

`mean` is the number usually quoted. Read the others too: a mean of a third of
a pixel with a maximum of three is a different calibration from one with a
maximum of half.

In [ ]:
rows = rep.summary()

In [ ]:
fig, ax = plt.subplots()
rep.plot_error_per_camera(ax=ax);

## 2. Which views are bad?

This is the plot to look at first, and the counterpart of MATLAB's
`showReprojectionErrors`. Each bar is one image.

A single view standing well above its neighbours is almost always one whose
corners were labelled the wrong way round. Remove that image and calibrate
again. A *group* of neighbouring views standing high is something else —
worth looking at the images themselves.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
rep.plot_error_per_view(ax=ax);

In [ ]:
# the worst views, as a table
rows = rep.per_view(printout=True)

## 3. Where in the frame does the error live?

Each arrow is one calibration point, drawn from where the corner was found
towards where the camera model puts it, exaggerated so it can be seen.

* arrows pointing every which way — measurement noise. This is what a good
  calibration looks like.
* arrows agreeing with their neighbours, growing towards the edges — lens
  distortion the model is not capturing.
* one patch of agreeing arrows in an otherwise random field — usually a
  misplaced point.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
rep.plot_error_over_frame(CAMS[0], ax=ax);

In [ ]:
# all the cameras side by side
fig, axs = plt.subplots(2, 2, figsize=(14, 9))
for a, c in zip(axs.ravel(), CAMS):
    rep.plot_error_over_frame(c, ax=a)
fig.tight_layout()

## 4. What part of the frame was calibrated?

Outside the region the points cover, the camera model is extrapolating, and
the error it shows on the points it was fitted to says nothing about how it
behaves there. If your particles will appear in a part of the frame the target
never visited, the calibration does not really cover them.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
rep.plot_coverage(ax=ax);

## 5. Where are the cameras?

The counterpart of MATLAB's `showExtrinsics`: each camera is drawn at its
recovered position with its field of view, and the calibration points are the
grey cloud.

Two things to check. That the arrangement is the arrangement of your actual
rig — this catches a camera reconstructed on the wrong side or pointing the
wrong way. And that the grey cloud covers your measurement volume, since that
is the region the calibration is supported in.

With `%matplotlib widget` above, this plot can be rotated with the mouse.

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection='3d')
rep.plot_extrinsics(ax=ax);

In [ ]:
# the distances between the cameras. These do not depend on the choice of lab
# frame, so if you know any of them from the rig, this is a real check.
import itertools
for a, b in itertools.combinations(CAMS, 2):
    d = np.linalg.norm(np.asarray(rep.cameras[a].O) - np.asarray(rep.cameras[b].O))
    print(f"{a} - {b}: {d:9.2f}")

## 6. Do the cameras agree with each other?

The reprojection error only says each camera fits its own points. This asks a
harder question: take the same physical corner as seen by every camera, and
see whether their rays actually meet. That is what the calibration is for.

The distance between the rays at their closest approach is in the units of
your lab coordinates.

In [ ]:
from myptv.imaging_mod import img_system

sysm = img_system([rep.cameras[c] for c in CAMS])

# gather the same lab position as seen by each camera
tab = {}
for i, c in enumerate(CAMS):
    for x, X in zip(rep.img[c], rep.lab[c]):
        tab.setdefault(tuple(np.round(X, 4)), {})[i] = tuple(x)

common = [k for k, v in tab.items() if len(v) >= 2]
print(f"{len(common)} points seen by two or more cameras")

d = []
for k in common:
    out = sysm.stereo_match(tab[k], 1e9)
    if out is not None:
        d.append(out[2])
d = np.array(d)
print(f"rays meet to within: median {np.median(d):.4f}, "
      f"p90 {np.percentile(d,90):.4f}, max {d.max():.4f}")

fig, ax = plt.subplots()
ax.hist(d, bins=60)
ax.set_xlabel('distance between the rays at their crossing [lab units]')
ax.set_ylabel('points')
ax.set_title('do the cameras agree?');

## 7. Removing a bad view

If section 2 showed one view far worse than the rest, this is how to check
what dropping it would do. It does **not** recalibrate — for that, remove the
image and run the calibration again — but it shows what the numbers would
look like without it.

In [ ]:
DROP = []          # e.g. [14] to see the effect of dropping view 14

if DROP:
    for c in CAMS:
        m = ~np.isin(rep.view[c], DROP)
        e_all = rep.errors(c)
        print(f"{c}: mean {e_all.mean():.4f} -> {e_all[m].mean():.4f} px "
              f"({(~m).sum()} points dropped)")
else:
    print("set DROP to a list of view numbers to try this")